# IntelliSense for a Relation Field

A relation field is an object. Its methods are already in it — the cell families the
arena contains, fixed the moment the substrate and token count are fixed. Nothing here
is invented; the tool reads what is present and reports what a position still needs.

The API mirrors an editor's, because the operations are the same ones:

| editor | here |
|---|---|
| `obj.` → list members | `field.methods()` |
| signature of a member | `field.signature(f)` — cells, rank, what it leaves open |
| what this position requires | `field.required(*declared)` |
| filter completions valid here | `field.candidates(*declared)` |
| go-to-definition for what is missing | the open dimension, by subtraction — no search |

One operation has no static form. Whether a method actually *closes* a position is not
determined by its signature: two methods can have the same type and occupy the same copy
of an irreducible rather than complementary ones. So `candidates` answers by evaluating,
and the evaluation is the certificate. **To read is to implement** — there is no separate
step between them.

Exact rational arithmetic throughout. The library cell below is complete and self-contained.

In [1]:
"""IntelliSense for a relation field. Self-contained, exact rational arithmetic."""
from fractions import Fraction
from itertools import combinations, permutations
from functools import lru_cache

# ---------------------------------------------------------------- exact linear algebra
def _rref(rows, nc):
    M=[[Fraction(x) for x in r] for r in rows]; piv=[]; rr=0
    for c in range(nc):
        p=next((i for i in range(rr,len(M)) if M[i][c]!=0), None)
        if p is None: continue
        M[rr],M[p]=M[p],M[rr]; pv=M[rr][c]; M[rr]=[x/pv for x in M[rr]]
        for i in range(len(M)):
            if i!=rr and M[i][c]!=0:
                f=M[i][c]; M[i]=[a-f*b for a,b in zip(M[i],M[rr])]
        piv.append(c); rr+=1
        if rr==len(M): break
    return M[:rr], piv
def rank(rows, nc):
    return 0 if not rows else len(_rref(rows,nc)[0])
def basis(rows, nc):
    return [] if not rows else _rref(rows,nc)[0]

# ---------------------------------------------------------------- substrate & arena
class Field:
    """The object. Its methods are the relation families already present in it."""
    def __init__(self, n, edges, K=2, group=None, name=None):
        self.n, self.K, self.name = n, K, name or f"graph({n}v,{len(edges)}e)"
        self.E = [tuple(sorted(e)) for e in edges]
        self.Eset = {frozenset(e) for e in self.E}
        self.G = group if group is not None else self._aut()
        self._build()

    def _aut(self):
        n=self.n; adj=[set() for _ in range(n)]
        for a,b in self.E: adj[a].add(b); adj[b].add(a)
        deg=[len(adj[v]) for v in range(n)]
        # refine by (degree, sorted neighbour degrees) to cut the search
        sig=[(deg[v], tuple(sorted(deg[u] for u in adj[v]))) for v in range(n)]
        order=sorted(range(n), key=lambda v:(-deg[v], v))
        out=[]; p=[-1]*n; used=[False]*n
        def bt(k):
            if k==n: out.append(tuple(p)); return
            v=order[k]
            for w in range(n):
                if used[w] or sig[w]!=sig[v]: continue
                ok=True
                for u in order[:k]:
                    if (u in adj[v]) != (p[u] in adj[w]): ok=False; break
                if ok:
                    p[v]=w; used[w]=True; bt(k+1); used[w]=False; p[v]=-1
        bt(0)
        return out

    def _build(self):
        n,K = self.n,self.K
        self.verts = sorted(combinations(range(n),K))
        vidx = {frozenset(s):i for i,s in enumerate(self.verts)}
        self.vidx = vidx
        ae=set()
        for i,S in enumerate(self.verts):
            s=set(S)
            for a in s:
                for b in range(n):
                    if b in s or frozenset((a,b)) not in self.Eset: continue
                    j=vidx[frozenset((s-{a})|{b})]
                    ae.add((min(i,j),max(i,j)))
        self.aedges=sorted(ae)
        self._cochain(); self._families()

    def _act(self,g,i):
        return self.vidx[frozenset(g[x] for x in self.verts[i])]

    def _cochain(self):
        oriented=[(a,b) for a,b in self.aedges]+[(b,a) for a,b in self.aedges]
        lab={}; bas=[]
        for oe in oriented:
            if oe in lab: continue
            orb={(self._act(g,oe[0]),self._act(g,oe[1])) for g in self.G}
            if (oe[1],oe[0]) in orb:
                for x in orb: lab[x]=(None,0)
            else:
                c=len(bas); bas.append(oe)
                for x in orb: lab[x]=(c,1)
                for x in orb: lab[(x[1],x[0])]=(c,-1)
        self.lab, self.dim1 = lab, len(bas)
        seen=set(); rows=[]
        for i in range(len(self.verts)):
            if i in seen: continue
            orb={self._act(g,i) for g in self.G}; seen|=orb
            rows.append([(1 if b in orb else 0)-(1 if a in orb else 0) for a,b in bas])
        self.r0 = rank(rows,self.dim1)
        self.D  = self.dim1 - self.r0          # the ambient capacity

    def _cycles(self,L):
        adj={}
        for a,b in self.aedges: adj.setdefault(a,set()).add(b); adj.setdefault(b,set()).add(a)
        out=set()
        def walk(start,cur,path):
            if len(path)==L:
                if start in adj.get(cur,()) and path[1]<path[-1]: out.add(tuple(path))
                return
            for nx in sorted(adj.get(cur,())):
                if nx<=start or nx in path: continue
                walk(start,nx,path+[nx])
        for s in sorted(adj): walk(s,s,[s])
        return [list(c) for c in out]

    def _row(self,cyc):
        v=[0]*self.dim1
        for i in range(len(cyc)):
            c,s=self.lab[(cyc[i],cyc[(i+1)%len(cyc)])]
            if c is not None: v[c]+=s
        return v

    def _families(self):
        V=self.verts
        def moves(c):
            S=[set(V[x]) for x in c]
            return [frozenset(((S[i]-S[(i+1)%len(c)]).pop(),(S[(i+1)%len(c)]-S[i]).pop()))
                    for i in range(len(c))]
        F={}
        for c in self._cycles(3):
            S=[set(V[x]) for x in c]
            F.setdefault("spec3" if set.intersection(*S) else "rot3",[]).append(c)
        for c in self._cycles(4):
            S=[set(V[x]) for x in c]
            k = "disj" if len(set(moves(c)))==2 else ("spec4" if set.intersection(*S) else "rot4")
            F.setdefault(k,[]).append(c)
        self.cells = {k:v for k,v in F.items() if v}
        self._b = {k: basis([self._row(c) for c in v], self.dim1) for k,v in self.cells.items()}

    # ------------------------------------------------------------ the IntelliSense API
    def methods(self):
        """Completion: the relation methods this field actually contains."""
        return sorted(self.cells, key=lambda k:-len(self.cells[k]))

    def signature(self, *fams):
        """The type of a method or a composition: cells, rank, and what it leaves open."""
        rows=[r for f in fams for r in self._b.get(f,[])]
        rk=rank(rows,self.dim1)
        return {"families":tuple(fams), "cells":sum(len(self.cells.get(f,[])) for f in fams),
                "rank":rk, "capacity":self.D, "open":self.D-rk}

    def required(self, *fams):
        """Go-to-definition for what is missing: the open dimension of a vocabulary."""
        return self.signature(*fams)["open"]

    def candidates(self, *fams):
        """Filter completions to those that close the position. Reading IS implementing:
        the rank evaluation that answers the question is the certificate."""
        have=set(fams); out=[]
        for f in self.methods():
            if f in have: continue
            s=self.signature(*(tuple(have)+(f,)))
            out.append({"method":f, "cells":len(self.cells[f]),
                        "closes":s["open"]==0, "remaining":s["open"]})
        return sorted(out, key=lambda d:(not d["closes"], d["remaining"], d["cells"]))

    def minimal_vocabularies(self):
        """The smallest method sets that close the field."""
        ms=self.methods(); sat=[]
        for k in range(1,len(ms)+1):
            for s in combinations(ms,k):
                if self.signature(*s)["open"]==0: sat.append(s)
            if sat: break
        return sat

    def report(self):
        L=[f"FIELD  {self.name}   K={self.K}",
           f"  arena {len(self.verts)} states, {len(self.aedges)} transitions",
           f"  |Aut| {len(self.G)}   invariant coords {self.dim1}   capacity D={self.D}",
           f"  methods present: {len(self.methods())}"]
        for f in self.methods():
            s=self.signature(f)
            L.append(f"    {f:<7s} {s['cells']:>5d} cells   rank {s['rank']:>3d}"
                     f"   leaves {s['open']:>3d} open")
        a=self.signature(*self.methods())
        L.append(f"  all methods together: rank {a['rank']} of {self.D}"
                 f"   -> field {'CLOSES' if a['open']==0 else 'leaves '+str(a['open'])+' open'}")
        mv=self.minimal_vocabularies()
        if mv: L.append(f"  minimal closing vocabularies: "
                        + "; ".join("+".join(s) for s in mv))
        return "\n".join(L)


## The object

`Field(n, edges, K)` builds the arena of `K` tokens on the graph, its invariant cochain
complex under a reader group (full automorphisms by default), and the relation families
present. `report()` is the summary an editor would show on hover.

In [2]:
W = (12, [(0,1),(0,2),(0,4),(0,5),(0,6),(0,7),(0,9),(1,2),(1,3),(1,5),(1,7),(1,8),
          (1,10),(2,3),(2,4),(2,8),(2,9),(2,11),(3,4),(3,5),(3,6),(3,9),(3,10),
          (4,5),(4,7),(4,10),(4,11),(5,6),(5,8),(5,11)])
f = Field(*W, K=2, name="witness: octahedron + six apexes")
print(f.report())

FIELD  witness: octahedron + six apexes   K=2
  arena 66 states, 300 transitions
  |Aut| 24   invariant coords 22   capacity D=15
  methods present: 5
    spec4     528 cells   rank   8   leaves   7 open
    disj      291 cells   rank  13   leaves   2 open
    rot4      264 cells   rank   7   leaves   8 open
    spec3     180 cells   rank  10   leaves   5 open
    rot3       20 cells   rank   2   leaves  13 open
  all methods together: rank 15 of 15   -> field CLOSES
  minimal closing vocabularies: disj+spec3; disj+rot3


## Completion at a position

Declare a vocabulary. The tool reports what the position still requires, then lists the
methods that are available and whether each one closes it. The ordering is the useful one:
closers first, then by how much they leave open, then by cost in cells.

In [3]:
print("declared: disj      requires:", f.required('disj'), "dimensions\n")
for c in f.candidates('disj'):
    print(f"  {c['method']:<7s} {c['cells']:>5d} cells   closes={str(c['closes']):<5s}  leaves {c['remaining']}")

declared: disj      requires: 2 dimensions

  rot3       20 cells   closes=True   leaves 0
  spec3     180 cells   closes=True   leaves 0
  rot4      264 cells   closes=False  leaves 1
  spec4     528 cells   closes=False  leaves 1


Two of the four candidates close it. The other two have enough reach on paper and fall
exactly one dimension short — that is the gap between a signature and a behaviour, and it
is why the list is produced by evaluation rather than by type-checking alone.

The cheapest closer is `rot3` at 20 cells against `spec3`'s 180. Nothing searched for it;
it is what remained after subtraction.

In [4]:
for v in f.minimal_vocabularies():
    s = f.signature(*v)
    print(f"  {'+'.join(v):<16s} {s['cells']:>5d} cells   rank {s['rank']}/{s['capacity']}")

  disj+spec3         471 cells   rank 15/15
  disj+rot3          311 cells   rank 15/15


## A field whose position cannot be completed

Point the same tool at the Petersen graph. Girth 5 means no triangles and no four-cycles
in the substrate, so four of the five method kinds do not exist in the arena at all. Read
with the trivial group so nothing is hidden by symmetry averaging.

In [5]:
P = (10, [(0,1),(1,2),(2,3),(3,4),(4,0),(5,7),(7,9),(9,6),(6,8),(8,5),
          (0,5),(1,6),(2,7),(3,8),(4,9)])
p = Field(*P, K=2, group=[tuple(range(10))], name="Petersen, trivial reader")
print(p.report())
print("\ncompletions available for the declared vocabulary 'disj':")
print("  ", p.candidates('disj') or "none — the field contains no other method")

FIELD  Petersen, trivial reader   K=2
  arena 45 states, 120 transitions
  |Aut| 1   invariant coords 120   capacity D=76
  methods present: 1
    disj       75 cells   rank  70   leaves   6 open
  all methods together: rank 70 of 76   -> field leaves 6 open

completions available for the declared vocabulary 'disj':
   none — the field contains no other method


Six dimensions open and an empty completion list. The position is well posed, its
requirement is computable, and nothing in the field answers it. This is the honest form of
an unfillable frame: not a gap in the field, but a method inventory that does not reach.

The contrast is the whole point of the tool. Both fields are complete objects. One has a
method that closes the position and one does not, and you find out by asking rather than
by proving.

## What this cannot do

`candidates` evaluates. It does not predict from the signature, and it cannot, because
multiplicity data records an inventory and not which copy of an irreducible a method
occupies — two submodules with identical signatures can coincide instead of spanning.
That is the same reason a type system never replaced a test suite.

The cost is bounded and does not grow with the reader lattice: one rank computation at any
reader that sees every channel, and the trivial reader always qualifies. A result certified
there holds at every other reader.